<a href="https://colab.research.google.com/github/Ntwenhle17/Sompisi-Town/blob/main/Phishing_Email_Detection_SVM_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import pandas as pd
import zipfile
import os

# Check which files have been uploaded
print("Files available:")
print(os.listdir("/content"))


Files available:
['.config', 'CEAS_08.csv.zip', 'sample_data']


In [3]:

import pandas as pd
import zipfile
import os

# Find the uploaded CSV or ZIP file
files = os.listdir("/content")
print("Uploaded files:", files)

# Load the dataset
zip_files = [f for f in files if f.lower().endswith(".zip")]
csv_files = [f for f in files if f.lower().endswith(".csv")]

if csv_files:
    file_path = "/content/" + csv_files[0]
    df = pd.read_csv(file_path, low_memory=False)
elif zip_files:
    zip_path = "/content/" + zip_files[0]

    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and not name.startswith("__MACOSX/")
        ]

        print("CSV files inside ZIP:", csv_names)

        if not csv_names:
            raise ValueError("No CSV file found inside the ZIP.")

        with archive.open(csv_names[0]) as file:
            df = pd.read_csv(file, low_memory=False)
else:
    raise FileNotFoundError(
        "Please upload your dataset as a CSV or ZIP file."
    )

print("\nDataset loaded successfully!")
print("Number of rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())


Uploaded files: ['.config', 'CEAS_08.csv.zip', 'sample_data']
CSV files inside ZIP: ['CEAS_08.csv']

Dataset loaded successfully!
Number of rows and columns: (39154, 7)

Column names:
['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']

First five rows:


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1



Missing values per column:
sender        0
receiver    462
date          0
subject      28
body          0
label         0
urls          0
dtype: int64

Number of duplicate rows:
0


In [4]:

# Step 5: Inspect the target labels

print("Label counts:")
print(df["label"].value_counts().sort_index())

print("\nLabel percentages:")
print(
    (df["label"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
    .astype(str) + "%"
)

print("\nURL flag counts:")
print(df["urls"].value_counts(dropna=False).sort_index())

print("\nURL flag by label:")
print(pd.crosstab(df["label"], df["urls"], margins=True))

print("\nExamples with label 0:")
display(df.loc[df["label"] == 0, ["subject", "body", "label"]].head(3))

print("\nExamples with label 1:")
display(df.loc[df["label"] == 1, ["subject", "body", "label"]].head(3))


Label counts:
label
0    17312
1    21842
Name: count, dtype: int64

Label percentages:
label
0    44.22%
1    55.78%
Name: proportion, dtype: object

URL flag counts:
urls
0    12922
1    26232
Name: count, dtype: int64

URL flag by label:
urls       0      1    All
label                     
0       5969  11343  17312
1       6953  14889  21842
All    12922  26232  39154

Examples with label 0:


,subject,body,label
3,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0
8,[Bug 5780] URI processing turns uuencoded stri...,http://issues.apache.org/SpamAssassin/show_bug...,0
15,RE: Trial IRC Certificate Application,"\nPlelim,\n\nJust to remind you that if a cert...",0



Examples with label 1:


,subject,body,label
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1


In [5]:

print("Examples with label 1 and no URL:")
display(
    df.loc[(df["label"] == 1) & (df["urls"] == 0),
           ["sender", "subject", "body"]].head(5)
)

print("\nExamples with label 0 and a URL:")
display(
    df.loc[(df["label"] == 0) & (df["urls"] == 1),
           ["sender", "subject", "body"]].head(5)
)


Examples with label 1 and no URL:


,sender,subject,body
5,Caroline Aragon <dwthaidomainnamesm@thaidomain...,From Caroline Aragon,\n\n\n\n\nYo wu urS mo ou go rc ebo eForM rgi ...
6,Replica Watches <jhorton@thebakercompanies.com>,Replica Watches,We have fake Swiss Men's and Ladie's Replica \...
25,dorian dominick <info@aamechanical.ca>,;) Look porno Gallery!!!,#PjWmcUDemi Moore Full video without cowards.\...
29,Maryellen Costello <shadiervh97@gillscruisecen...,MBA-Degreees not for sale- get it.,"Bacheelor, MasteerMBA, and Doctoraate diplomas..."
31,Cara Childs <xigoing@cityofws.org>,Man's stuff store,\n\n\nLargest selection of medication products...



Examples with label 0 and a URL:


,sender,subject,body
3,Michael Parker <ivqrnai@pobox.com>,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...
8,qydlqcws-iacfym@issues.apache.org,[Bug 5780] URI processing turns uuencoded stri...,http://issues.apache.org/SpamAssassin/show_bug...
15,Racing <uqyrmo@sailing.ie>,RE: Trial IRC Certificate Application,"\nPlelim,\n\nJust to remind you that if a cert..."
18,Aaron Kulkis <cmiqlkx91@hotpop.com>,"Re: [opensuse] Why can't I use ""shutdown now"" ...",Carlos E. R. wrote: > -----BEGIN PGP SIGNED ME...
19,Aaron Kulkis <cmiqlkx91@hotpop.com>,Re: Fwd: [opensuse] Re: openSUSE Boxed Editions,Steve Jacobs wrote: > ---------- Forwarded mes...


In [1]:

!pip -q install datasets

from datasets import load_dataset

dataset = load_dataset("drorrabin/phishing_emails-data")

print(dataset)
print("\nAvailable splits:")
print(dataset.keys())

first_split = next(iter(dataset.keys()))
print("\nColumn names:")
print(dataset[first_split].column_names)

print("\nFirst three examples:")
print(dataset[first_split][:3])


README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.88MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26946 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3705 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})

Available splits:
dict_keys(['train', 'test'])

Column names:
['text', 'email_type']

First three examples:
{'text': ['Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/

In [2]:

import pandas as pd

# Convert the Hugging Face splits into pandas DataFrames
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining label counts:")
print(train_df["email_type"].value_counts())

print("\nTest label counts:")
print(test_df["email_type"].value_counts())

print("\nMissing values in training data:")
print(train_df.isnull().sum())

print("\nDuplicate text rows in training data:")
print(train_df["text"].duplicated().sum())

print("\nUnique labels:")
print(train_df["email_type"].unique())

print("\nFirst 300 characters of one email:")
print(train_df["text"].iloc[0][:300])


Training shape: (26946, 2)
Test shape: (3705, 2)

Training label counts:
email_type
phishing email    13473
safe email        13473
Name: count, dtype: int64

Test label counts:
email_type
safe email        3369
phishing email     336
Name: count, dtype: int64

Missing values in training data:
text          0
email_type    0
dtype: int64

Duplicate text rows in training data:
0

Unique labels:
['phishing email' 'safe email']

First 300 characters of one email:
Is the following email safe or phishing??

Date: Thu, 07 Aug 2008 10:59:35 +0200

Sender: Daily Top 10 <avadivap_1981@techsult.com>

Receiver: user2.12@gvc.ceas-challenge.cc

Email Subject: CNN.com Daily Top 10

Email Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:


In [3]:

# Check class proportions in both splits
print("Training class proportions:")
print(train_df["email_type"].value_counts(normalize=True).mul(100).round(2))

print("\nTest class proportions:")
print(test_df["email_type"].value_counts(normalize=True).mul(100).round(2))

# Check for label hints in the text
for name, frame in [("Training", train_df), ("Test", test_df)]:
    text = frame["text"].fillna("").str.lower()

    print(f"\n{name}:")
    print(
        "'email type is:' occurrences:",
        text.str.contains("email type is:", regex=False).sum()
    )
    print(
        "'safe or phishing' occurrences:",
        text.str.contains("safe or phishing", regex=False).sum()
    )
    print(
        "'phishing email' occurrences:",
        text.str.contains("phishing email", regex=False).sum()
    )
    print(
        "'safe email' occurrences:",
        text.str.contains("safe email", regex=False).sum()
    )


Training class proportions:
email_type
phishing email    50.0
safe email        50.0
Name: proportion, dtype: float64

Test class proportions:
email_type
safe email        90.93
phishing email     9.07
Name: proportion, dtype: float64

Training:
'email type is:' occurrences: 26946
'safe or phishing' occurrences: 26946
'phishing email' occurrences: 13474
'safe email' occurrences: 13473

Test:
'email type is:' occurrences: 3705
'safe or phishing' occurrences: 3705
'phishing email' occurrences: 0
'safe email' occurrences: 0


In [4]:

# Display the endings of three training examples
for i in range(3):
    print(f"\n--- Example {i + 1} ---")
    print(train_df["text"].iloc[i][-250:])
    print("True label:", train_df["email_type"].iloc[i])



--- Example 1 ---
 http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08

Email type is: phishing email
True label: phishing email

--- Example 2 ---
think about your health & personal life.

And we are glad to aid you with it.

 From now on till 1st of  October you can use our special proposal.

Check our site for further details.

answerdry.com

6 Aug 2008 09:49:01

Email type is: phishing email
True label: phishing email

--- Example 3 ---
hat response from that server
 and retry.

What you want is the default in postfix 2.5. You'll have to look
at "smtp_sasl_auth_soft_bounce". 



/Per Jessen, Zürich

-- 
http://www.spamchek.com/ - your spam is our business.

Email type is: safe email
True label: safe email


In [5]:

import re
import pandas as pd
from sklearn.model_selection import train_test_split

# Make copies so the original data stays unchanged
data = train_df.copy()

# Remove the artificial question at the beginning
def clean_email(text):
    text = str(text)

    text = re.sub(
        r"^\s*Is the following email safe or phishing\?\?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove the artificial label appended to training emails
    text = re.sub(
        r"\s*Email type is:\s*(?:phishing email|safe email)\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

data["clean_text"] = data["text"].apply(clean_email)

# Convert labels to numeric values
data["label"] = data["email_type"].map({
    "safe email": 0,
    "phishing email": 1
})

# Check for labels that could not be mapped
if data["label"].isna().any():
    raise ValueError("Unexpected label found. Check email_type values.")

data["label"] = data["label"].astype(int)

# Create a stratified 80/20 split
X_train, X_val, y_train, y_val = train_test_split(
    data["clean_text"],
    data["label"],
    test_size=0.20,
    random_state=42,
    stratify=data["label"]
)

print("Cleaning completed.")
print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))

print("\nTraining class proportions (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation class proportions (%):")
print((y_val.value_counts(normalize=True) * 100).round(2))

print("\nExample of cleaned text:")
print(X_train.iloc[0][:500])
print("\nLabel:", y_train.iloc[0])


Cleaning completed.
Training examples: 21556
Validation examples: 5390

Training class proportions (%):
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Validation class proportions (%):
label
1    50.0
0    50.0
Name: proportion, dtype: float64

Example of cleaned text:
Date: Wed, 06 Aug 2008 16:00:25 -0400

Sender: Barry Warsaw <pjaxq@python.org>

Receiver: "Martin v. Löwis" <qpnysl@v.loewis.de>

Email Subject: Re: [Python-Dev] 2.6 and 3.0 project management

Email Body: -----BEGIN PGP SIGNED MESSAGE-----
Hash: SHA1

On Mar 17, 2008, at 12:27 AM, Martin v. Löwis wrote:
 'critical' is fine (or 'immediate').  My problem before was that I   
 couldn't do one query that gave me all the critical issues for  
 both  2.6 and 3.0.  That certainly could have been pebka

Label: 0


In [6]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Convert email text into numerical features
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

# Fit only on training data, then transform validation data
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print("TF-IDF conversion completed.")
print("Training feature matrix:", X_train_tfidf.shape)
print("Validation feature matrix:", X_val_tfidf.shape)
print("Number of vocabulary terms:", len(tfidf.vocabulary_))

# Show a few learned vocabulary terms
print("\nSample vocabulary terms:")
print(list(tfidf.vocabulary_.keys())[:20])


TF-IDF conversion completed.
Training feature matrix: (21556, 20000)
Validation feature matrix: (5390, 20000)
Number of vocabulary terms: 20000

Sample vocabulary terms:
['date', 'wed', '06', 'aug', '2008', '16', '00', '25', '0400', 'sender', 'barry', 'warsaw', 'pjaxq', 'python', 'org', 'receiver', 'martin', 'lowis', 'qpnysl', 'loewis']


In [7]:

from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import time

# Model 1: Support Vector Machine
svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=3000
)

start = time.time()
svm_model.fit(X_train_tfidf, y_train)
svm_time = time.time() - start

svm_predictions = svm_model.predict(X_val_tfidf)

# Model 2: XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

start = time.time()
xgb_model.fit(X_train_tfidf, y_train)
xgb_time = time.time() - start

xgb_predictions = xgb_model.predict(X_val_tfidf)

# Calculate evaluation metrics
results = []

for name, predictions, train_time in [
    ("SVM", svm_predictions, svm_time),
    ("XGBoost", xgb_predictions, xgb_time)
]:
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1-score": f1_score(y_val, predictions, zero_division=0),
        "Training time (seconds)": round(train_time, 2)
    })

results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(results_df.round(4))

print("\nSVM Classification Report")
print(classification_report(
    y_val, svm_predictions,
    labels=[0, 1],
    target_names=["Safe email", "Phishing email"],
    zero_division=0
))

print("\nXGBoost Classification Report")
print(classification_report(
    y_val, xgb_predictions,
    labels=[0, 1],
    target_names=["Safe email", "Phishing email"],
    zero_division=0
))

print("\nSVM Confusion Matrix")
print(confusion_matrix(y_val, svm_predictions, labels=[0, 1]))

print("\nXGBoost Confusion Matrix")
print(confusion_matrix(y_val, xgb_predictions, labels=[0, 1]))


MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1-score,Training time (seconds)
0,SVM,0.9985,0.9993,0.9978,0.9985,1.05
1,XGBoost,0.9959,0.9956,0.9963,0.9959,137.96



SVM Classification Report
                precision    recall  f1-score   support

    Safe email       1.00      1.00      1.00      2695
Phishing email       1.00      1.00      1.00      2695

      accuracy                           1.00      5390
     macro avg       1.00      1.00      1.00      5390
  weighted avg       1.00      1.00      1.00      5390


XGBoost Classification Report
                precision    recall  f1-score   support

    Safe email       1.00      1.00      1.00      2695
Phishing email       1.00      1.00      1.00      2695

      accuracy                           1.00      5390
     macro avg       1.00      1.00      1.00      5390
  weighted avg       1.00      1.00      1.00      5390


SVM Confusion Matrix
[[2693    2]
 [   6 2689]]

XGBoost Confusion Matrix
[[2683   12]
 [  10 2685]]


In [8]:

# Audit random examples from both classes
for label, name in [(0, "SAFE EMAIL"), (1, "PHISHING-LABELLED EMAIL")]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    samples = data[data["label"] == label].sample(
        n=5, random_state=42
    )

    for i, (_, row) in enumerate(samples.iterrows(), start=1):
        print(f"\nExample {i}")
        print("Label:", row["email_type"])
        print(row["clean_text"][:700])
        print("-" * 50)

# Check whether any artificial label hints remain
clean_lower = data["clean_text"].str.lower()

print("\nRemaining artificial label statements:")
print(
    clean_lower.str.contains(
        r"email type is:\s*(phishing email|safe email)",
        regex=True
    ).sum()
)

print("\nRemaining introductory prompts:")
print(
    clean_lower.str.contains(
        "is the following email safe or phishing",
        regex=False
    ).sum()
)



==================== SAFE EMAIL ====================

Example 1
Label: safe email
Date: Wed, 06 Aug 2008 23:05:16 +0100

Sender: James Holderness <v8_cgejz@hotmail.com>

Receiver: Sam Ruby <iyqxo@intertwingly.net>

Email Subject: Re: Planet venus decode error

Email Body: Sam Ruby wrote:
 You managed to break the Universal Feed Parser.  Congratulations.  :-)

w00t! So do I win a t-shirt or something? ;)

 A (possibly partial) fix has been committed, but in order to exhibit these 
 symptoms I suspect that there are more.

Thanks, that fixed it for me. Although for some reason it managed to butcher 
the contents of my most recent post, but that's another issue altogether.

Regards
James

-- 
devel mailing list
kpdpo@lists.planetplanet.org
http://lists.planetplanet.org/mailm
--------------------------------------------------

Example 2
Label: safe email
Date: Tue, 05 Aug 2008 16:10:38 -0800

Sender: qydlqcws-iacfym@issues.apache.org

Receiver: xrh@spamassassin.apache.org

Email Subject: 

/tmp/ipykernel_3900/2047401825.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  clean_lower.str.contains(


In [9]:

from datasets import load_dataset

# Display the dataset's available metadata
info = dataset["train"].info

print("Dataset description:")
print(info.description or "No dataset description available.")

print("\nDataset citation:")
print(info.citation or "No citation supplied in this metadata.")

print("\nDataset features:")
print(dataset["train"].features)


Dataset description:
No dataset description available.

Dataset citation:
No citation supplied in this metadata.

Dataset features:
{'text': Value('string'), 'email_type': Value('string')}


In [10]:

print("TRAIN LABEL COUNTS")
print(dataset["train"].to_pandas()["email_type"].value_counts())

print("\nTEST LABEL COUNTS")
print(dataset["test"].to_pandas()["email_type"].value_counts())

print("\nEXAMPLE TEST EMAILS")
for i in range(3):
    row = dataset["test"][i]
    print("\nLABEL:", row["email_type"])
    print(row["text"][:700])


TRAIN LABEL COUNTS
email_type
phishing email    13473
safe email        13473
Name: count, dtype: int64

TEST LABEL COUNTS
email_type
safe email        3369
phishing email     336
Name: count, dtype: int64

EXAMPLE TEST EMAILS

LABEL: safe email
Is the following email safe or phishing??

Date: Wed, 06 Aug 2008 22:21:00 -0500

Sender: Brandon Black <auwhylg@gmail.com>

Receiver: Linda W <twax-vptwszv@tlinx.org>, Perl 5 Porters <qtog3-epoawmh@perl.org>

Email Subject: Re: Broken Perl OO-parent-method call

Email Body: On 10/26/07, Brandon Black  wrote:
 Most likely the cleanest way to fix this without breaking anything
 else is to make the method lookup code ignore these stub entries and
 look past them in the inheritance lookup, if they're easy to identify.


Except I wasn't thinking about AUTOLOAD.  I bet the above would break
AUTOLOAD.  I never use it, so I never really think of it.  Hmmm.

-- Brandon

Email type is: 

LABEL: safe email
Is the following email safe or phishing??

Date:

In [11]:

phishing_examples = data[data["label"] == 1]["clean_text"]

for i, email in enumerate(phishing_examples.sample(5, random_state=42), start=1):
    print(f"\n--- PHISHING-LABELLED EMAIL {i} ---")
    print(email[:1000])



--- PHISHING-LABELLED EMAIL 1 ---
Date: Wed, 06 Aug 2008 02:56:47 +0200

Sender: Guy Huff <gerencia-ig@jfw.com.br>

Receiver: user3@gvc.ceas-challenge.cc

Email Subject: Katerina age 29 -on dating 

Email Body: -------------------------------------------------
Dating
Katerina age 29  From: Logan, Utah, United States of America:
http://enylakitafa.narod.ru/?qKDA2dNFhv0ZLkw0ZL

--- PHISHING-LABELLED EMAIL 2 ---
Date: Fri, 08 Aug 2008 03:54:12 -0400

Sender: <>

Receiver: user2.1-ext1@gvc.ceas-challenge.cc

Email Subject: Nadal caught with prostitutes

Email Body: Prescription data used to judge consumers http://www.bedwade.com/

--- PHISHING-LABELLED EMAIL 3 ---
Date: Thu, 07 Aug 2008 10:26:07 -0600

Sender: Madelyn Rivera <hsrgarlic@badischer-judo-verband.de>

Receiver: user3@gvc.ceas-challenge.cc

Email Subject: If you are looking for

Email Body: World of health, 80% cut.


Great variety of products for your health.

Here!




fImultiple epiphanous equitorial
familistic euryhaline es

In [12]:

phishing_examples = data[data["label"] == 1]["clean_text"]

for i, email in enumerate(phishing_examples.sample(5, random_state=42), start=1):
    print(f"\n--- PHISHING-LABELLED EMAIL {i} ---")
    print(email[:1000])



--- PHISHING-LABELLED EMAIL 1 ---
Date: Wed, 06 Aug 2008 02:56:47 +0200

Sender: Guy Huff <gerencia-ig@jfw.com.br>

Receiver: user3@gvc.ceas-challenge.cc

Email Subject: Katerina age 29 -on dating 

Email Body: -------------------------------------------------
Dating
Katerina age 29  From: Logan, Utah, United States of America:
http://enylakitafa.narod.ru/?qKDA2dNFhv0ZLkw0ZL

--- PHISHING-LABELLED EMAIL 2 ---
Date: Fri, 08 Aug 2008 03:54:12 -0400

Sender: <>

Receiver: user2.1-ext1@gvc.ceas-challenge.cc

Email Subject: Nadal caught with prostitutes

Email Body: Prescription data used to judge consumers http://www.bedwade.com/

--- PHISHING-LABELLED EMAIL 3 ---
Date: Thu, 07 Aug 2008 10:26:07 -0600

Sender: Madelyn Rivera <hsrgarlic@badischer-judo-verband.de>

Receiver: user3@gvc.ceas-challenge.cc

Email Subject: If you are looking for

Email Body: World of health, 80% cut.


Great variety of products for your health.

Here!




fImultiple epiphanous equitorial
familistic euryhaline es